# 29 · Triage diagnosis: train the student once

Runs step 5 of [`docs/experiments/triage_diagnosis/PROTOCOL.md`](../docs/experiments/triage_diagnosis/PROTOCOL.md)
(version 6), locked before any student existed, in about **10–20 minutes**. Use a **T4 GPU runtime**
(Runtime → Change runtime type → T4 GPU).

It checks out the pinned code commit, fetches the Booking CSV and checks its SHA-256, rebuilds every review text
from it and stops unless each matches the committed keys, then trains DistilBERT once on the teacher's 10,000
labels with the settings version 6 fixes, and writes the student's logits for the development, test and check
reviews.

Nothing is scored here: thresholds and scores are later, local steps. No API key and no Hugging Face token are
needed.

**Send back** `triage_student_results.zip`. It holds the logits and `run.json`, not the model weights.

In [ ]:
from pathlib import Path

SOURCE_COMMIT = "dba52d0340055d752ca6fe40ce5ec11ba60ad35f"
REPO = Path("/content/hotel-review-nlp")
OUT = Path("docs/experiments/triage_diagnosis/student")  # inside the repository: logits and run.json only

In [ ]:
import os
import subprocess
import sys

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", "https://github.com/krimits/hotel-review-nlp.git", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(["git", "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "checkout", "--quiet", "--detach", SOURCE_COMMIT], check=True)
GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if GIT_COMMIT != SOURCE_COMMIT:
    raise RuntimeError(f"checked out {GIT_COMMIT}, not the pinned {SOURCE_COMMIT}")
sys.path.insert(0, str(REPO / "src"))
# The transformers version of the project's other runs and of the CI.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "transformers==4.56.2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", "."], check=True)
print("commit", GIT_COMMIT)

## 1. Data: the Booking CSV the samples were drawn from

In [ ]:
import torch


def run(command: str) -> None:
    print("$", command, flush=True)
    subprocess.run(command, shell=True, check=True)


if not torch.cuda.is_available():
    raise RuntimeError("No GPU: choose Runtime -> Change runtime type -> T4 GPU, then run all again.")
run("python scripts/fetch_booking_515k.py")  # refuses a file that is not the Kaggle download
print("GPU", torch.cuda.get_device_name(0))

## 2. Train once (about 5–15 minutes)

The script rebuilds every text and stops unless all 10,900 match their keys; then it trains for 4 epochs and
writes the logits. It refuses to run twice into the same folder.

In [ ]:
run("python scripts/train_triage_student.py --csv data/raw/booking_reviews_515k.csv")

## 3. Collect and download

In [ ]:
import hashlib
import json
import zipfile

collected = sorted(path for path in OUT.iterdir() if path.is_file() and path.name != "artifact_manifest.json")
if any(path.suffix in (".safetensors", ".bin", ".pt") for path in collected):
    raise RuntimeError("model weights in the bundle")
names = {path.name for path in collected}
expected = {"run.json", "logits_development.csv", "logits_test.csv", "logits_check.csv"}
if names != expected:
    raise RuntimeError(f"the bundle holds {sorted(names)}, not {sorted(expected)}")
manifest = {"commit": GIT_COMMIT,
            "files": [{"path": path.as_posix(), "bytes": path.stat().st_size,
                       "sha256": hashlib.sha256(path.read_bytes()).hexdigest()} for path in collected]}
(OUT / "artifact_manifest.json").write_text(json.dumps(manifest, indent=1) + "\n", encoding="utf-8")
with zipfile.ZipFile("triage_student_results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in [*collected, OUT / "artifact_manifest.json"]:
        archive.write(path, path.as_posix())
print(json.loads((OUT / "run.json").read_text(encoding="utf-8"))["loss_per_epoch"], f"{len(collected) + 1} files")

In [ ]:
from google.colab import files

files.download("triage_student_results.zip")

## 4. After the run

Send `triage_student_results.zip` back. Its paths are the repository's: the files are checked against
`artifact_manifest.json` and the pinned commit, and committed. Then, locally: the thresholds are chosen on the
development set and committed, and only after that is the test set scored, once.